# Itemset on Google Colab

Run all cells on a CPU runtime. The notebook resolves the public repository to an exact commit, verifies Julia 1.13.2, Java 25 and the pinned SPMF JAR, then runs the Julia demo. Setup and application data downloads can take several minutes.

The demo checks the package, mines baskets and Semeion images, and compares both variants with an independent SPMF run on Chess. The Chess check measures peak process RSS for one threshold. The full benchmark series have their own commands and figures in the [report guide](https://github.com/NgTHung/Itemset/tree/main/docs/report).

In [ ]:
import os
from pathlib import Path
import re
import runpy
import subprocess
import tempfile
import urllib.request

repository = "https://github.com/NgTHung/Itemset"
revision = os.environ.get("ITEMSET_REVISION")
if not revision:
    revision = subprocess.check_output(
        ["git", "ls-remote", "--exit-code", repository, "refs/heads/main"],
        text=True,
    ).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("ITEMSET_REVISION must be a full Git commit SHA")

workspace = Path(tempfile.mkdtemp(prefix="itemset-colab-", dir="/content"))
runner_path = workspace / "run_colab_notebook.py"
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/NgTHung/Itemset/{revision}/scripts/run_colab_notebook.py",
    runner_path,
)
evidence = workspace / "evidence"
evidence_zip = evidence.with_suffix(".zip")
print(f"Revision: {revision}", flush=True)
print(f"Evidence ZIP: {evidence_zip}", flush=True)
runner = runpy.run_path(str(runner_path))
if runner["main"]([
    "--revision", revision,
    "--root", str(workspace / "repo"),
    "--evidence", str(evidence),
]):
    raise RuntimeError(f"Demo failed. Open {evidence_zip} for the log.")

## Semeion patterns

The examples below compare source images that contain a visual-word pattern with images that lack one or more of its words. Use the Colab Files sidebar to download the evidence ZIP. It includes the full itemset maps, source image IDs, basket rules, SPMF comparisons and peak RSS measurements.

In [ ]:
from IPython.display import SVG, display

semeion = evidence / "semeion"
display(SVG(filename=str(semeion / "visual_words.svg")))
display(SVG(filename=str(semeion / "supporting-0.05.svg")))
print(f"Evidence ZIP: {evidence_zip}")